In [0]:
from pyspark.sql.functions import col

# Read the Customers table from default schema
df_customers_gold = spark.table("workspace.my_database.customers_gold")

# Read the Orders Gold table
df_orders_gold = spark.table("workspace.my_database.orders_gold")

# Perform a LEFT JOIN based on 'customer_id'
df_customer_orders_left = df_customers_gold.join(
    df_orders_gold,
    on="customer_id",
    how="left"
)

# Fill nulls in order columns for customers who haven't placed any order
df_customer_orders_final = df_customer_orders_left.fillna({
    "quantity": 0,
    "discount": 0,
    "order_size": "No Order",
    "is_priority_order": False
})

# Save the CLEANED dataframe (df_customer_orders_final) as a table
df_customer_orders_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("my_database.customers_orders_master")

print("Left Join successful: Nulls filled and Master table created!")

In [0]:
# Create or replace a permanent view based on the master table
spark.sql("""
    CREATE OR REPLACE VIEW workspace.my_database.vw_customers_orders_master AS
    SELECT * 
    FROM workspace.my_database.customers_orders_master
""")

print("View 'vw_customers_orders_master' has been successfully created!")

# Read the view into a DataFrame to verify the data
df_master_view = spark.table("workspace.my_database.vw_customers_orders_master")

# Display the view
display(df_master_view)